# M3-B1 — Mission étoile : préparer la 3ᵉ source pour du RAG — SUJET

> **Bonus optionnel, non noté.** Acerox t'a transmis un corpus de **rapports
> techniques** (`rapports_techniques_2024.zip`, 5 fichiers `.md`). C'est du
> **texte non structuré** : impossible à ranger dans une table SQL. Tu vas le
> **préparer** pour une exploitation par similarité sémantique.

> 📚 **Avant de commencer**, lis [`../ressources/06_RAG_prepa_essentiel.md`](../ressources/06_RAG_prepa_essentiel.md)
> (~25 min) : les 4 concepts (chunk, embedding, similarité, base vectorielle) +
> un exemple qui tourne sans le zip.

## 🚫 Garde-fou
> **Préparation seulement** : tu t'arrêtes à la **récupération** (retrieval).
> La génération augmentée par un LLM, c'est **M7-B2**. Pour la prédiction de
> défauts (tabulaire), un RAG **n'aide pas** : ce corpus sert l'**aide au
> diagnostic humain**. Embeddings **locaux** (aucune clé API).

## Pré-requis
```bash
# Dézippe rapports_techniques_2024.zip dans  ../data/rapports_techniques_2024/
# Décommente le bloc bonus de requirements.txt puis :
pip install chromadb sentence-transformers
```

## 1. Repérer les rapports (fourni)

In [50]:
from pathlib import Path

RAPPORTS = Path("../data/rapports_techniques_2024")
EMBED_MODEL = "all-MiniLM-L6-v2"  # Adapté au corpus ? (regarde la langue des rapports)

fichiers = sorted(RAPPORTS.glob("*.md"))
print(f"{len(fichiers)} rapports trouvés :")
for f in fichiers:
    print(" -", f.name)

5 rapports trouvés :
 - RT-2026-001_Roubaix_ligne1.md
 - RT-2026-002_Lyon_ligne2.md
 - RT-2026-003_Lyon_ligne4.md
 - RT-2026-004_Lyon_ligne4.md
 - RT-2026-005_Lyon_ligne2.md


## 2. DocumentLoader (fait main)

Écris une fonction qui lit un `.md` et renvoie un dict avec au moins :
`source`, `reference`, `date` (extraits de l'en-tête par regex) et `texte`.
Pas besoin de LangChain.

In [51]:
import re


def charger_rapport(path: Path) -> dict:
    contenu = path.read_text(encoding="utf-8")
    reference = re.search(r"Référence\s*:\s*(\S+)", contenu).group(1)
    date = re.search(r"Date\s*:\s*(\d{4}-\d{2}-\d{2})", contenu).group(1)
    return {
        "source": path.name,
        "reference": reference,
        "date": date,
        "texte": contenu,
    }


docs = [charger_rapport(f) for f in fichiers]
for d in docs:
    print(d["reference"], d["date"], d["source"])

RT-2026-001 2026-04-01 RT-2026-001_Roubaix_ligne1.md
RT-2026-002 2026-04-08 RT-2026-002_Lyon_ligne2.md
RT-2026-003 2026-04-11 RT-2026-003_Lyon_ligne4.md
RT-2026-004 2026-04-18 RT-2026-004_Lyon_ligne4.md
RT-2026-005 2026-04-28 RT-2026-005_Lyon_ligne2.md


## 3. Segmentation (chunking)

Implémente **2 stratégies** et **justifie ton choix** :
- **par titre Markdown** (`##`) : chunks cohérents avec la structure ;
- **par taille fixe** (avec recouvrement) : chunks homogènes.

> C'est l'item syllabus « implémentation de techniques de segmentation ».

In [52]:
def chunk_par_titre(doc: dict) -> list[dict]:
    meta_base = {k: doc[k] for k in ("source", "reference", "date")}
    sections = re.split(r"^## ", doc["texte"], flags=re.MULTILINE)[1:]
    chunks = []
    for i, sec in enumerate(sections):
        titre = sec.splitlines()[0].strip()
        chunks.append({
            "id": f"{doc['reference']}-t{i}",
            "texte": "## " + sec.strip(),
            "meta": {**meta_base, "section": titre},
        })
    return chunks


def chunk_taille_fixe(doc: dict, taille: int = 500, overlap: int = 80):
    meta_base = {k: doc[k] for k in ("source", "reference", "date")}
    texte = doc["texte"]
    chunks = []
    start, i = 0, 0
    while start < len(texte):
        morceau = texte[start:start + taille]
        chunks.append({
            "id": f"{doc['reference']}-f{i}",
            "texte": morceau,
            "meta": {**meta_base, "section": None},
        })
        start += taille - overlap
        i += 1
    return chunks


chunks_titre = [c for d in docs for c in chunk_par_titre(d)]
chunks_fixe = [c for d in docs for c in chunk_taille_fixe(d)]
print("par titre :", len(chunks_titre), "chunks")
print("taille fixe :", len(chunks_fixe), "chunks")

par titre : 10 chunks
taille fixe : 10 chunks


In [53]:
print("=== PAR TITRE ===")
print("section :", chunks_titre[0]["meta"]["section"])
print(chunks_titre[0]["texte"])
print("\n=== TAILLE FIXE ===")
print(chunks_fixe[0]["texte"])

=== PAR TITRE ===
section : Dérive thermique ligne 1
## Dérive thermique ligne 1

Synthèse hebdomadaire des alertes supervision.

Température moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'échangeur. Intervention de nettoyage planifiée, retour à la normale sous 48 h.

=== TAILLE FIXE ===
# Rapport technique — Roubaix

> Référence : RT-2026-001  |  Site : Roubaix  |  Ligne : 1  |  Date : 2026-04-01  |  Équipement : capteur de température S12

## Dérive thermique ligne 1

Synthèse hebdomadaire des alertes supervision.

Température moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'échangeur. Intervention de nettoyage planifiée, retour à la normale sous 48 h.

## Suite donnée

- Responsable : EMP-2424
- Statut : en cours
- 


**Choix : découpage par titre.** Chaque chunk correspond alors à une section entière portant sur un seul sujet, c'est-à-dire une unité de sens cohérente. Le découpage à taille fixe, lui, coupe au milieu des sections et mélange plusieurs sujets dans un même chunk, ce qui dégrade la pertinence de la recherche par le sens.

## 4. Embeddings + indexation ChromaDB

Encode chaque chunk avec `SentenceTransformer(EMBED_MODEL)` (local), puis range
les chunks dans une collection ChromaDB **persistante**.

> Pense à **vider** la collection si elle existe déjà (idempotence du notebook).

In [54]:
import chromadb
from sentence_transformers import SentenceTransformer

chunks = chunks_titre

modele = SentenceTransformer(EMBED_MODEL)
embeddings = modele.encode([c["texte"] for c in chunks]).tolist()

client = chromadb.PersistentClient(path="./chroma_acerox")
try:
    client.delete_collection("rapports_acerox")
except Exception:
    pass
col = client.create_collection("rapports_acerox")

col.add(
    ids=[c["id"] for c in chunks],
    documents=[c["texte"] for c in chunks],
    embeddings=embeddings,
    metadatas=[c["meta"] for c in chunks],
)
print(col.count(), "chunks indexés")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


10 chunks indexés


## 5. Interroger par similarité

In [56]:
def interroger(question: str, k: int = 3) -> None:
    res = col.query(query_embeddings=modele.encode([question]).tolist(), n_results=k)
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"[{dist:.3f}] {meta['reference']} — {meta['section']}")
        print(doc)
        print("-" * 60)


# Exemples de questions métier à tester :
# interroger("Pourquoi des défauts sur la ligne de laminage ?")
interroger("Quels capteurs sont peu fiables ?")
# interroger("Risque RGPD avec les données opérateurs de l'ERP ?")

[0.945] RT-2026-005 — Incident capteur S02
## Incident capteur S02

Note technique transmise au service industrialisation.

Trous de données (65 relevés manquants) sur le S02 entre le 2026-04-28 et le lendemain. Cause : perte de connexion réseau du concentrateur. Données interpolées non utilisées pour le scoring.
------------------------------------------------------------
[1.147] RT-2026-003 — Baisse de débit ligne 4
## Baisse de débit ligne 4

Note technique transmise au service industrialisation.

Débit moyen tombé à 91 u/h (nominal 120 u/h). Colmatage partiel suspecté en amont. Purge effectuée par EMP-6881, débit restauré à 95 %.
------------------------------------------------------------
[1.157] RT-2026-004 — Suite donnée
## Suite donnée

- Responsable : EMP-1750
- Statut : clôturé
- Lien supervision : voir `capteurs_iot.csv` (2026-04-18, site Lyon, line_id 4).
------------------------------------------------------------


## 6. Relationnel vs vectoriel

**Relationnel** (capteurs, ERP) : des données structurées en lignes/colonnes, stockées en base SQL et interrogées par critères exacts (filtres, agrégats, calculs).

**Vectoriel** (rapports techniques) : du texte non structuré transformé en embeddings et stocké en base vectorielle, interrogé par similarité de sens plutôt que par mots exacts.

**Conclusion** : chaque type de stockage correspond à une nature de donnée — le relationnel pour les valeurs précises des capteurs et de l'ERP, le vectoriel pour le texte libre des rapports interrogé par le sens.

## ✅ Vérification (checklist)

- [ ] J'ai chargé les 5 rapports + leurs métadonnées (loader fait main)
- [ ] J'ai comparé 2 stratégies de **segmentation** et justifié mon choix
- [ ] J'ai indexé les chunks dans ChromaDB avec des embeddings **locaux**
- [ ] Une requête en langage naturel récupère les bons passages
- [ ] J'ai vérifié que le **modèle d'embedding est adapté à la langue du corpus** (les rapports sont en français)
- [ ] J'ai écrit 3 lignes (dans ma note) sur **relationnel vs vectoriel**
- [ ] Je sais où s'arrête la **préparation** vs le **RAG complet** (M7-B2)

## ⭐ Pour aller plus loin
- Filtrer par métadonnée (`where={...}`) pour ne chercher que les rapports récents.
- Brancher un LLM local (Ollama) sur les chunks — mais : est-ce justifié ici ?